# Multiple polynomiale Regression – **B: Individuelle Polynomgrade je Feature**

**Ziel:** Für jede numerische Eingabevariable den Polynomgrad separat festlegen, die Regressionsmodelle trainieren und die Ergebnisse mit Trainings-/Validierungs-RMSE und Grafiken vergleichen.

**Wichtige Unterschiede zu Variante A:** Wir erzeugen Potenzen **je Feature getrennt**. Kreuzprodukte/Interaktionen zwischen verschiedenen Features werden **nicht automatisch** erzeugt. Ein Polynomgrad von 2 für `age` bedeutet, dass sowohl `age` als auch `age²` in das Modell eingehen; das Modell schätzt die Koeffizienten selbst.

**Ablauf:** CSV hochladen → Variablen erkunden → individuelle Polynomgrade definieren → Modelle vergleichen → Feature-Kurven betrachten → einmalige Testbewertung.


## 1. CSV-Datensatz hochladen

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(f'Datensatz: {filename} | Zeilen: {len(df)} | Spalten: {len(df.columns)}')
display(df.head())

## 2. Numerische Variablen ansehen

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()
print('Numerische Variablen:')
print(', '.join(numeric_columns))

## 3. Features und ihre Polynomgrade wählen

**Hier bearbeiten:** Jeder Schlüssel in `FEATURE_DEGREES` ist ein Eingabe-Feature; die Zahl gibt den **maximalen Grad für genau dieses Feature** an. Tragen Sie nur Spaltennamen aus der CSV ein. Zum Beispiel 1 = linear, 2 = quadratisch, 3 = kubisch.

`MODEL_VARIANTS` enthält optionale Vergleichsmodelle. Für eine erste Übung genügt das selbst definierte Modell; die Beispielalternativen können Sie aktiv lassen oder entfernen.

In [ ]:
# KONFIGURATION – hier die individuellen Polynomgrade ändern
Y_VARIABLE = 'marketvalue'
FEATURE_DEGREES = {
    'goalsTotal': 1,
    'age': 2,
    'wikipedia': 3,
}

# Optional: weitere Hypothesenräume mit denselben Features vergleichen
MODEL_VARIANTS = {
    'Eigene Hypothese': FEATURE_DEGREES,
    'Alle linear': {feature: 1 for feature in FEATURE_DEGREES},
    'Alle quadratisch': {feature: 2 for feature in FEATURE_DEGREES},
}

assert FEATURE_DEGREES, 'Mindestens ein Feature auswählen.'
assert all(isinstance(d, int) and not isinstance(d, bool) and 1 <= d <= 6
           for d in FEATURE_DEGREES.values()), 'Erlaubte Polynomgrade: ganze Zahlen von 1 bis 6.'
X_VARIABLES = list(FEATURE_DEGREES)
assert Y_VARIABLE not in X_VARIABLES, 'Die Zielvariable darf nicht als Feature verwendet werden.'
assert all(set(v) == set(X_VARIABLES) for v in MODEL_VARIANTS.values()),     'Alle Modellvarianten müssen dieselben Features enthalten.'
assert all(all(isinstance(d, int) and not isinstance(d, bool) and 1 <= d <= 6
               for d in v.values()) for v in MODEL_VARIANTS.values()),     'Alle Modellvarianten müssen Grade zwischen 1 und 6 verwenden.'
missing = [c for c in X_VARIABLES + [Y_VARIABLE] if c not in df.columns]
assert not missing, f'Fehlende Spalten: {missing}'
non_numeric = [c for c in X_VARIABLES + [Y_VARIABLE]
               if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f'Nicht numerische Spalten: {non_numeric}'

model_df = df[X_VARIABLES + [Y_VARIABLE]].replace([np.inf, -np.inf], np.nan).dropna().copy()
assert len(model_df) >= 30, 'Zu wenige vollständige Datenpunkte (mindestens 30 erforderlich).'
print(f'{len(model_df)} vollständige Beobachtungen für das Modell.')
display(pd.DataFrame({'Feature': X_VARIABLES,
                      'Polynomgrad': [FEATURE_DEGREES[f] for f in X_VARIABLES]}))

## 4. Explorative Scatterplots: Welcher Zusammenhang erscheint plausibel?

Die Scatterplots zeigen jeweils nur **einen** Zusammenhang. Die spätere Regression nutzt alle Features gemeinsam. Die beobachteten Formen sind zunächst Hypothesen, keine Beweise.

In [ ]:
import matplotlib.pyplot as plt

for feature in X_VARIABLES:
    plt.figure(figsize=(7, 4))
    plt.scatter(model_df[feature], model_df[Y_VARIABLE], alpha=0.25, s=14)
    plt.xlabel(feature)
    plt.ylabel(Y_VARIABLE)
    plt.title(f'{Y_VARIABLE} vs. {feature} | gewählter Grad: {FEATURE_DEGREES[feature]}')
    plt.grid(alpha=0.2)
    plt.show()

## 5. Trainings-, Validierungs- und Testdaten erzeugen

60 % Training, 20 % Validierung, 20 % Test. Die Testdaten bleiben bis zum Schluss unberührt.

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df[X_VARIABLES]
y = model_df[Y_VARIABLE]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)
print(f'Training: {len(X_train)} | Validierung: {len(X_val)} | Test: {len(X_test)}')

## 6. Regressionsmodell aufbauen: **Polynome pro Feature**

`ColumnTransformer` verarbeitet jedes Feature separat; `PolynomialFeatures` erzeugt je Feature Potenzen bis zu dessen Grad. `StandardScaler` standardisiert **jedes Feature anhand der Trainingsdaten** vor den Potenzen. Deshalb beziehen sich die später angezeigten Koeffizienten auf standardisierte Variablen `z_feature = (feature − Mittelwert) / Standardabweichung`.

**Modell ohne Interaktionen:** \(\hat y=eta_0+\sum_j\sum_{k=1}^{d_j}eta_{jk} z_j^k\).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def build_model(feature_degrees):
    transformations = []
    for feature, degree in feature_degrees.items():
        individual = Pipeline([
            ('scale', StandardScaler()),
            ('powers', PolynomialFeatures(degree=degree, include_bias=False)),
        ])
        transformations.append((feature, individual, [feature]))
    return Pipeline([
        ('features', ColumnTransformer(transformations, remainder='drop')),
        ('regression', LinearRegression()),
    ])

print('Individuell definierte Terme (vereinfachte Schreibweise):')
for feature, degree in FEATURE_DEGREES.items():
    print(f'  {feature}: ' + ', '.join(feature if k == 1 else f'{feature}^{k}'
                                      for k in range(1, degree + 1)))
print('Interaktionsterme zwischen Features: keine')

## 7. Hypothesenräume trainieren und Fehler vergleichen

Die **Validierungsdaten** bestimmen die beste Modellvariante. Ein besonders niedriger Trainingsfehler ist allein kein Qualitätsnachweis.

In [ ]:
results = []
models = {}
for name, degrees in MODEL_VARIANTS.items():
    model = build_model(degrees)
    model.fit(X_train, y_train)
    models[name] = model
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
    results.append({'Modell': name,
                    'Polynomgrade': ', '.join(f'{f}: {d}' for f, d in degrees.items()),
                    'Anzahl Terme': sum(degrees.values()),
                    'RMSE Training': train_rmse,
                    'RMSE Validierung': val_rmse})
results_df = pd.DataFrame(results).sort_values('RMSE Validierung').reset_index(drop=True)
display(results_df)
best_name = results_df.loc[0, 'Modell']
print(f'Beste Variante nach Validierungs-RMSE: {best_name}')

## 8. Einfluss jedes Features visualisieren

**Ein Diagramm pro Feature.** Auf der x-Achse wird jeweils nur ein Feature verändert; alle anderen Features werden auf den **Medianen der Trainingsdaten** festgehalten. Die Kurve ist daher eine bedingte Modellvorhersage, nicht die vollständige mehrdimensionale Funktion. Scatterpunkte zeigen dagegen den ungefilterten bivariaten Zusammenhang.

In [ ]:
selected_model = models['Eigene Hypothese']  # Für Kurven der eigenen Grade
reference = X_train.median(numeric_only=True)
for feature in X_VARIABLES:
    x_values = np.linspace(X_train[feature].min(), X_train[feature].max(), 250)
    X_grid = pd.DataFrame([reference.to_dict()] * len(x_values), columns=X_VARIABLES)
    X_grid[feature] = x_values
    predicted = selected_model.predict(X_grid)
    plt.figure(figsize=(8, 4.7))
    plt.scatter(X_train[feature], y_train, alpha=0.2, s=14,
                label='Trainingsdaten (bivariat)')
    plt.plot(x_values, predicted, linewidth=2.5,
             label=f'Modellvorhersage (Grad {FEATURE_DEGREES[feature]})')
    plt.xlabel(feature)
    plt.ylabel(Y_VARIABLE)
    plt.title(f'{feature}: bedingter Modellverlauf; andere Features = Trainingsmedian')
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()

## 9. Trainings- und Validierungsfehler vergleichen

In [ ]:
plot_df = results_df.set_index('Modell')[['RMSE Training', 'RMSE Validierung']]
ax = plot_df.plot(kind='bar', figsize=(9, 5), rot=15)
ax.set_ylabel('RMSE (Einheit der Zielvariable)')
ax.set_title('Vergleich individueller Polynomgrade')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

## 10. Koeffizienten und geschätzte Modellgleichung betrachten

Zur Interpretation werden die Koeffizienten zusammen mit ihren Potenzen ausgegeben. **Beachte:** Wegen der Skalierung beziehen sich die Koeffizienten auf standardisierte Merkmale (`z_feature`), nicht unmittelbar auf Rohwerte.

In [ ]:
reg = selected_model.named_steps['regression']
feature_transformer = selected_model.named_steps['features']
terms = []
for feature, degree in FEATURE_DEGREES.items():
    for k in range(1, degree + 1):
        terms.append(f'z_{feature}' if k == 1 else f'z_{feature}^{k}')
coefficients = pd.DataFrame({'Term (standardisiert)': terms,
                             'Koeffizient': reg.coef_})
display(coefficients)
print(f'Intercept: {reg.intercept_:.4f}')
print('Modellgleichung (gerundet; z = (x - Trainingsmittelwert)/Trainingsstandardabweichung):')
print('ŷ = ' + f'{reg.intercept_:.4g}' + ''.join(
    f' {coef:+.4g}·{term}' for term, coef in zip(terms, reg.coef_)))

## 11. Finale Bewertung auf unberührten Testdaten

Erst nach der Modellwahl werden Training und Validierung zusammengeführt und das ausgewählte Modell neu geschätzt. Verwenden Sie die Testdaten **nicht**, um wiederholt neue Grade auszuwählen.

In [ ]:
winning_degrees = MODEL_VARIANTS[best_name]
final_model = build_model(winning_degrees)
final_model.fit(X_dev, y_dev)
final_pred = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
print(f'Gewählte Variante: {best_name}')
print('Grade:', winning_degrees)
print(f'RMSE Testdaten: {test_rmse:.3f}')

## 12. Reflexionsfragen

1. Welche Polynomgrade haben Sie für die einzelnen Features gewählt? Begründen Sie die Entscheidung anhand der Scatterplots.
2. Wie unterscheiden sich die Trainings- und Validierungs-RMSE der Modellvarianten?
3. Ist ein höherer Polynomgrad immer besser? Wo erkennen Sie Hinweise auf Overfitting oder Underfitting?
4. Warum erscheinen in der Modellgleichung sowohl `age` als auch `age²`, wenn für `age` der Grad 2 gewählt wird?
5. Warum ergeben die einzelnen Feature-Grafiken **nicht** die gesamte Modellfunktion?
6. Was ist der Unterschied zwischen **individuellen Polynomgraden ohne Interaktionen** (dieses Notebook) und einem **gemeinsamen Gesamtgrad mit Interaktionen** (Variante A)?
7. Welche Veränderungen würden Sie testen, ohne den Testsatz zur Modellwahl zu nutzen?

**Hinweis zur Interpretation:** Die Form der Kurven hängt von den Daten und den geschätzten Koeffizienten ab. Ein quadratischer Term muss nicht zwingend einen deutlich sichtbaren U-förmigen Verlauf ergeben. Multikollinearität und extreme Extrapolation können die Interpretation erschweren.
